In [1]:
!pip install xlrd

In [28]:
import boto3
import pandas as pd
from io import BytesIO
from pathlib import Path
import re

In [15]:
bucket_name = "proyecto-electricidad-ela"
prefix_clima = "bronze/clima/"  

s3 = boto3.client("s3")

silver_output = Path("../data/silver/clima/clima.parquet")
silver_output.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
dfs = []

for key in xls_keys:
    try:
        response_file = s3.get_object(Bucket=bucket_name, Key=key)
        content = response_file["Body"].read()

        df = pd.read_excel(BytesIO(content), engine="xlrd", header=4)

        df["source_file"] = key
        dfs.append(df)

        print(f"OK: {key} | filas: {len(df)}")

    except Exception as e:
        print(f"ERROR en {key}: {type(e).__name__} - {e}")

print("Número de dataframes cargados:", len(dfs))

In [27]:
df_clima_raw = pd.concat(dfs, ignore_index=True)
df_clima_raw.head()

,Estación,Provincia,Temperatura máxima (ºC),Temperatura mínima (ºC),Temperatura media (ºC),Racha (km/h),Velocidad máxima (km/h),Precipitación 00-24h (mm),Precipitación 00-06h (mm),Precipitación 06-12h (mm),Precipitación 12-18h (mm),Precipitación 18-24h (mm),source_file
0,Alforja,Tarragona,13.6 (15:20),4.0 (23:10),8.8,41 (00:10),24 (12:50),0.0,0.0,0.0,0.0,0.0,bronze/clima/Aemet2024-01-01.xls
1,Reus Aeropuerto,Tarragona,15.3 (13:40),2.7 (23:59),9.0,21 (00:40),14 (00:40),0.0,0.0,0.0,0.0,0.0,bronze/clima/Aemet2024-01-01.xls
2,Valls,Tarragona,15.3 (13:50),2.3 (23:20),8.8,NaN,NaN,0.0,0.0,0.0,0.0,0.0,bronze/clima/Aemet2024-01-01.xls
3,Tarragona,Tarragona,15.5 (13:30),5.9 (23:59),10.7,NaN,NaN,0.0,0.0,0.0,0.0,0.0,bronze/clima/Aemet2024-01-01.xls
4,Pontons,Barcelona,12.4 (15:50),3.4 (23:59),7.9,32 (00:10),21 (00:10),0.0,0.0,0.0,0.0,0.0,bronze/clima/Aemet2024-01-01.xls


In [29]:
df_clima_raw.columns = (
    df_clima_raw.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [32]:
print(df_clima_raw.columns.tolist())

['estación', 'provincia', 'temperatura_máxima_c', 'temperatura_mínima_c', 'temperatura_media_c', 'racha_km/h', 'velocidad_máxima_km/h', 'precipitación_00-24h_mm', 'precipitación_00-06h_mm', 'precipitación_06-12h_mm', 'precipitación_12-18h_mm', 'precipitación_18-24h_mm', 'source_file']


In [34]:
df_clima = df_clima_raw[[
    "temperatura_media_c",
    "precipitación_00-24h_mm",
    "velocidad_máxima_km/h",
    "source_file"
]].copy()

df_clima.rename(columns={
    "temperatura_media_c": "temperatura_media",
    "precipitación_00-24h_mm": "precipitacion",
    "velocidad_máxima_km/h": "viento"
}, inplace=True)

In [35]:
def limpiar_valor(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    x = re.sub(r"\s*\(.*\)", "", x)
    x = x.replace(",", ".")
    try:
        return float(x)
    except:
        return None

In [36]:
df_clima["temperatura_media"] = df_clima["temperatura_media"].apply(limpiar_valor)
df_clima["precipitacion"] = df_clima["precipitacion"].apply(limpiar_valor)
df_clima["viento"] = df_clima["viento"].apply(limpiar_valor)

In [38]:
df_clima.isnull().sum()

temperatura_media     25475
precipitacion         33101
viento               100577
source_file               0
dtype: int64

In [43]:
df_clima["fecha"] = df_clima["source_file"].str.extract(r'(\d{4}-\d{2}-\d{2})')
df_clima["fecha"] = pd.to_datetime(df_clima["fecha"])

In [44]:
df_clima["temperatura_media"] = df_clima["temperatura_media"].fillna(
    df_clima["temperatura_media"].mean()
)

df_clima["precipitacion"] = df_clima["precipitacion"].fillna(0)

df_clima["viento"] = df_clima["viento"].fillna(
    df_clima["viento"].mean()
)

df_clima.isnull().sum()

temperatura_media    0
precipitacion        0
viento               0
source_file          0
fecha                0
dtype: int64

In [46]:
df_clima_diario = (
    df_clima
    .groupby("fecha", as_index=False)
    .agg({
        "temperatura_media": "mean",
        "precipitacion": "mean",
        "viento": "mean"
    })
)

In [47]:
df_clima.head()

,temperatura_media,precipitacion,viento,source_file,fecha
0,8.8,0.0,24.00000,bronze/clima/Aemet2024-01-01.xls,2024-01-01
1,9.0,0.0,14.00000,bronze/clima/Aemet2024-01-01.xls,2024-01-01
2,8.8,0.0,20.09187,bronze/clima/Aemet2024-01-01.xls,2024-01-01
3,10.7,0.0,20.09187,bronze/clima/Aemet2024-01-01.xls,2024-01-01
4,7.9,0.0,21.00000,bronze/clima/Aemet2024-01-01.xls,2024-01-01


In [48]:
df_clima_diario.to_parquet("../data/silver/clima/clima.parquet", index=False)